# Group C — Theoretical Guarantees

Verifies the **two core correctness guarantees** that ASCAL must satisfy at every
snapshot (after every positive demonstration).

| Guarantee | Condition | Why it holds |
|---|---|---|
| **C1** Sound precision ≡ 1.0 | `p_s == 1.0` | `L_pre` only accumulates literals confirmed by positive pre-states — can never fire in a failed state |
| **C2** Complete recall ≡ 1.0 | `r_c == 1.0` | `U_pre` starts as `{∅}` (always applicable); UUP only adds distinguishing literals — the true model always remains inside the version space |

### How evaluation is done

`run_group_c` calls `learner.evaluate(test_pos, test_neg)` at every positive-demo
snapshot and asserts both conditions above. This maps to the values
`evaluate_detailed` computes:

- **C1** is checked via `precision_sound` — equals 1.0 iff `fp_sound == 0`,
  i.e., `L_pre` never fires on a negative test demo.
- **C2** is checked via `recall_complete` — equals 1.0 iff `fn_complete == 0`,
  i.e., every positive test demo is covered by at least one version-space hypothesis.

`evaluate_detailed` uses the **minimal-witness** approach for the complete-model
positive check (see `AlgorithmExplained.md §11b`): instead of enumerating all
effect hypotheses between `L_eff` and `U_eff` — which was O(2^N) and caused
multi-GB memory usage — it checks existence via `he* = (L_eff − hp) ∪ delta`
in O(|U_pre|) per demo. Memory is now constant per snapshot.

### Key design choice: held-out test set

Using training demos as the test set trivially satisfies C1 (the sound model was
built from those demos, so it can never FP on them). A genuinely **held-out 20 %
test set** is required for the check to have real diagnostic value.

The demos are split **before** any learning: 80 % training, 20 % test.
Both splits preserve the positive/negative ratio.

### Memory strategy

The blocks domain generates up to 68,000 raw negative demos per problem before
lifting deduplication. Two mitigations keep memory bounded:

1. **`max_neg_per_step=50`** inside `generate_demos` — at each plan step, at most
   50 inapplicable groundings are sampled. Across 20 problems this still yields
   all ~1,692 unique lifted negatives after deduplication.
2. **Incremental deduplication** in the collection loop — unique demos are
   tracked with a `seen` set; duplicates are discarded immediately, so memory
   never holds more than the number of unique demos.

Reference: `AlgorithmExplained.md §11b, §13 Group C`

In [1]:
import sys, os, tempfile, shutil, random as _random

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from unified_planning.shortcuts import *
from unified_planning.io import PDDLReader
import unified_planning
get_environment().credits_stream = None

from ascal.learner import Learner
from ascal.models import Literal, State, Action, Demonstration
from ascal.transitions import generate_lifted_demonstrations_from_problem

## Setup — Mockup Domain

In [2]:
MOCKUP_DIR   = os.path.join('..', 'benchmarks', 'mockup')
DOMAIN_FILE  = os.path.join(MOCKUP_DIR, 'domain.pddl')
PROBLEM_FILE = os.path.join(MOCKUP_DIR, 'problems', 'problem-00.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
assert os.path.exists(PROBLEM_FILE), f'Problem not found: {PROBLEM_FILE}'
print(f'Domain:  {DOMAIN_FILE}')
print(f'Problem: {PROBLEM_FILE}')

Domain:  ../benchmarks/mockup/domain.pddl
Problem: ../benchmarks/mockup/problems/problem-00.pddl


In [3]:
reader  = PDDLReader()
problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)

all_fluents    = problem.fluents
all_actions    = problem.actions
static_fluents = problem.get_static_fluents()
action_names   = [a.name for a in all_actions]

print('Actions :', action_names)
print('Fluents :', [f.name for f in all_fluents])
print('Static  :', [f.name for f in static_fluents])

Actions : ['pickup']
Fluents : ['on', 'on_table', 'clear', 'arm_empty', 'holding']
Static  : ['on']


## Helpers

In [4]:
def generate_demos(prob, max_neg_per_step=50, seed=0):
    """Delegates to :func:`ascal.transitions.generate_lifted_demonstrations_from_problem`."""
    return generate_lifted_demonstrations_from_problem(
        prob,
        max_neg_per_step=max_neg_per_step,
        max_check_per_action=None,
        seed=seed,
        verbose=False,
    )


def split_demos(pos_list, neg_list, train_ratio=0.8, seed=42):
    """Split pos and neg lists independently (preserving ratio).

    Returns (train_pos, train_neg, test_pos, test_neg).
    """
    rng = _random.Random(seed)

    def _split(lst):
        shuffled = list(lst)
        rng.shuffle(shuffled)
        cut = max(1, int(len(shuffled) * train_ratio))
        return shuffled[:cut], shuffled[cut:]

    train_pos, test_pos = _split(pos_list)
    train_neg, test_neg = _split(neg_list)
    return train_pos, train_neg, test_pos, test_neg


print('generate_demos() and split_demos() defined.')

generate_demos() and split_demos() defined.


In [5]:
tmpdir   = tempfile.mkdtemp(prefix='ascal_testC_')
orig_cwd = os.getcwd()
os.chdir(tmpdir)
try:
    pos_demos, neg_demos = generate_demos(problem)
finally:
    os.chdir(orig_cwd)
    shutil.rmtree(tmpdir, ignore_errors=True)

print(f'Total: {len(pos_demos)} positive, {len(neg_demos)} negative')

Total: 1 positive, 0 negative


In [6]:
train_pos, train_neg, test_pos, test_neg = split_demos(pos_demos, neg_demos)

print(f'Training : {len(train_pos)} pos, {len(train_neg)} neg')
print(f'Test     : {len(test_pos)} pos, {len(test_neg)} neg')

if not test_pos:
    print('⚠  No positive test demos — C2 check will be vacuous.')
    print('   Run the blocks section below for a meaningful test.')

Training : 1 pos, 0 neg
Test     : 0 pos, 0 neg
⚠  No positive test demos — C2 check will be vacuous.
   Run the blocks section below for a meaningful test.


## Main Learning Loop — Guarantee Checks at Every Positive Demo

At every snapshot (after every **positive** training demo), `run_group_c` calls
`learner.evaluate(test_pos, test_neg)` and asserts:

```
C1: p_s == 1.0   (sound precision — no FP on negative test demos)
C2: r_c == 1.0   (complete recall — every positive test demo covered)
```

If either assertion fails, the function prints the failing snapshot index, the
current boundaries, and the specific FP/FN that caused the violation.

> **Note:** `evaluate_detailed()` was updated (2026-03) to use the minimal-witness
> approach for the complete-model positive check, replacing a 2^N enumeration
> with O(|U_pre|) subset checks. Memory is constant per snapshot.

In [7]:
def fmt_lits(fset):
    if not fset:
        return '\u2205'
    return '{' + ', '.join(sorted(str(l) for l in fset)) + '}'


def run_group_c(fluents, actions, static, train_pos, train_neg, test_pos, test_neg,
                label=''):
    """Run Group C guarantee checks at every positive-demo snapshot.

    At each snapshot, calls BOTH learner.evaluate() and learner.evaluate_repr()
    and checks:

      C1: p_s  == 1.0  (sound precision — guaranteed by algorithm)
      C2: r_c  == 1.0  (complete recall — guaranteed by algorithm)

    Both methods should satisfy C1/C2.  If evaluate_repr() violates C2 where
    evaluate() does not, it means the single most-general representative is not
    always consistent with positive test demos (the algorithm's guarantee only
    covers the existential — any hypothesis in U_pre — not a specific one).

    Returns (snapshots_passed, snapshots_failed) based on evaluate() (the primary
    method); repr violations are flagged but do not stop the loop.
    """
    learner = Learner(fluents, actions, static)

    if not train_pos:
        all_train = list(train_neg)
    elif not train_neg:
        all_train = list(train_pos)
    else:
        slice_size = len(train_neg) / len(train_pos)
        all_train  = []
        for i, pos in enumerate(train_pos):
            all_train.extend(train_neg[int(slice_size * i):int(slice_size * (i + 1))])
            all_train.append(pos)

    snapshot_idx = 0
    passed = failed = 0
    repr_c2_failures = 0   # count snapshots where evaluate_repr C2 fails

    if label:
        print(f'\n=== {label} ===')
    print(f'Training: {len(train_pos)} pos + {len(train_neg)} neg ({len(all_train)} total)')
    print(f'Test    : {len(test_pos)} pos + {len(test_neg)} neg')
    print(f'{"Snap":>5} {"Demo":>5}  {"evaluate() P_s":>14} {"R_c":>6}  {"evaluate_repr() P_s":>19} {"R_c":>6}')
    print('-' * 70)

    for demo_idx, demo in enumerate(all_train):
        learner.update(demo)
        if demo.post_state is None:
            continue
        if not test_pos and not test_neg:
            snapshot_idx += 1
            continue

        # Primary method
        f1_s,  f1_c,  p_s,  r_s,  p_c,  r_c         = learner.evaluate(test_pos, test_neg)
        # Representative method
        f1_s2, f1_c2, p_s2, r_s2, p_c2, r_c2, status = learner.evaluate_repr(test_pos, test_neg)

        c1_ok      = abs(p_s  - 1.0) < 1e-9
        c2_ok      = abs(r_c  - 1.0) < 1e-9
        c1_ok_repr = abs(p_s2 - 1.0) < 1e-9
        c2_ok_repr = abs(r_c2 - 1.0) < 1e-9

        sym_c1      = '\u2713' if c1_ok      else '\u274c'
        sym_c2      = '\u2713' if c2_ok      else '\u274c'
        sym_c1_repr = '\u2713' if c1_ok_repr else '\u274c'
        sym_c2_repr = '\u2713' if c2_ok_repr else '\u274c'

        print(f'{snapshot_idx:>5} {demo_idx:>5}  '
              f'P_s={p_s:.4f}{sym_c1} R_c={r_c:.4f}{sym_c2}  '
              f'P_s={p_s2:.4f}{sym_c1_repr} R_c={r_c2:.4f}{sym_c2_repr}')

        if c1_ok and c2_ok:
            passed += 1
        else:
            failed += 1
            print('  \u26a0  VIOLATION (evaluate)')
            if not c1_ok:
                print(f'  C1 FAILED: Sound precision = {p_s:.6f} (expected 1.0)')
                for a in actions:
                    n = a.name
                    print(f'    L_pre[{n}] = {fmt_lits(next(iter(learner.L_pre[n])))}')
            if not c2_ok:
                print(f'  C2 FAILED: Complete recall = {r_c:.6f} (expected 1.0)')
                for a in actions:
                    n = a.name
                    print(f'    U_pre[{n}] ({len(learner.U_pre[n])} hyps):')
                    for hp in learner.U_pre[n]:
                        print(f'      {fmt_lits(hp)}')
            break

        if not c2_ok_repr:
            repr_c2_failures += 1

        snapshot_idx += 1

    print(f'\n{"=" * 70}')
    if failed == 0:
        print(f'\u2705 evaluate()      — ALL {passed} snapshots: C1 and C2 hold.')
        if repr_c2_failures == 0:
            print(f'\u2705 evaluate_repr() — ALL {passed} snapshots: C1 and C2 hold.')
        else:
            print(f'\u26a0  evaluate_repr() — C2 (complete recall) failed at {repr_c2_failures}/{passed} snapshot(s).')
            print('    This means the single most-general representative hypothesis was not')
            print('    always consistent with positive test demos.')
            print("    Note: this is expected before convergence — the algorithm's C2 guarantee")
            print('    covers "any hypothesis in U_pre", not "the specific min-length one".')
    else:
        print(f'\u274c VIOLATION at snapshot {snapshot_idx}. {passed} snapshots passed before failure.')

    return passed, failed


print('run_group_c() defined.')


run_group_c() defined.


In [8]:
run_group_c(
    all_fluents, all_actions, static_fluents,
    train_pos, train_neg, test_pos, test_neg,
    label='Mockup domain'
)


=== Mockup domain ===
Training: 1 pos + 0 neg (1 total)
Test    : 0 pos + 0 neg
 Snap  Demo  evaluate() P_s    R_c  evaluate_repr() P_s    R_c
----------------------------------------------------------------------

✅ evaluate()      — ALL 0 snapshots: C1 and C2 hold.
✅ evaluate_repr() — ALL 0 snapshots: C1 and C2 hold.


(0, 0)

## Scale-up: Blocks Domain (4 actions, 20 problems)

The mockup domain has too few demos for a meaningful held-out test set.
The blocks domain with 20 problems produces ~67 unique positive and ~1,692 unique
negative lifted demos — enough for a proper stress test of both guarantees.

In [9]:
BLOCKS_DIR    = os.path.join('..', 'benchmarks', 'blocks')
BLOCKS_DOMAIN = os.path.join(BLOCKS_DIR, 'domain_extended.pddl')
BLOCKS_PROBS  = os.path.join(BLOCKS_DIR, 'problems')

assert os.path.exists(BLOCKS_DOMAIN), f'Blocks domain not found: {BLOCKS_DOMAIN}'

problem_files = sorted(f for f in os.listdir(BLOCKS_PROBS) if f.endswith('.pddl'))
print(f'Blocks domain : {BLOCKS_DOMAIN}')
print(f'Problem files : {len(problem_files)}')

blocks_reader  = PDDLReader()
blocks_problem = blocks_reader.parse_problem(
    BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, problem_files[0]))

b_fluents      = blocks_problem.fluents
b_actions      = blocks_problem.actions
b_static       = blocks_problem.get_static_fluents()
b_action_names = [a.name for a in b_actions]
print(f'Actions       : {b_action_names}')

Blocks domain : ../benchmarks/blocks/domain_extended.pddl
Problem files : 21
Actions       : ['pickup', 'putdown', 'stack', 'unstack']


In [10]:
# Collect unique lifted demos across all 20 problems.
# Deduplication is incremental — duplicates are discarded immediately so memory
# is bounded by the number of unique demos, not the raw generation count.
seen_pos, seen_neg     = set(), set()
all_pos_blocks, all_neg_blocks = [], []

for pf in problem_files:
    prob = blocks_reader.parse_problem(BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, pf))
    tmpdir   = tempfile.mkdtemp(prefix='ascal_testC_blocks_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        p, n = generate_demos(prob)   # max_neg_per_step=50 by default
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)

    new_p = new_n = 0
    for d in p:
        k = repr(d)
        if k not in seen_pos:
            seen_pos.add(k)
            all_pos_blocks.append(d)
            new_p += 1
    for d in n:
        k = repr(d)
        if k not in seen_neg:
            seen_neg.add(k)
            all_neg_blocks.append(d)
            new_n += 1

    print(f'  {pf}: +{new_p} new pos  +{new_n} new neg  '
          f'(running unique: {len(all_pos_blocks)} pos, {len(all_neg_blocks)} neg)')

print(f'\nFinal unique: {len(all_pos_blocks)} pos, {len(all_neg_blocks)} neg')

  problem-00.pddl: +13 new pos  +548 new neg  (running unique: 13 pos, 548 neg)
  problem-01.pddl: +9 new pos  +318 new neg  (running unique: 22 pos, 866 neg)
  problem-02.pddl: +10 new pos  +266 new neg  (running unique: 32 pos, 1132 neg)
  problem-03.pddl: +12 new pos  +113 new neg  (running unique: 44 pos, 1245 neg)
  problem-04.pddl: +13 new pos  +261 new neg  (running unique: 57 pos, 1506 neg)
  problem-05.pddl: +4 new pos  +47 new neg  (running unique: 61 pos, 1553 neg)
  problem-06.pddl: +1 new pos  +39 new neg  (running unique: 62 pos, 1592 neg)
  problem-07.pddl: +0 new pos  +20 new neg  (running unique: 62 pos, 1612 neg)
  problem-08.pddl: +1 new pos  +30 new neg  (running unique: 63 pos, 1642 neg)
  problem-09.pddl: +2 new pos  +23 new neg  (running unique: 65 pos, 1665 neg)
  problem-10.pddl: +0 new pos  +3 new neg  (running unique: 65 pos, 1668 neg)
  problem-11.pddl: +0 new pos  +0 new neg  (running unique: 65 pos, 1668 neg)
  problem-12.pddl: +0 new pos  +0 new neg  (run

In [11]:
b_train_pos, b_train_neg, b_test_pos, b_test_neg = split_demos(
    all_pos_blocks, all_neg_blocks, train_ratio=0.8)

print(f'Training : {len(b_train_pos)} pos, {len(b_train_neg)} neg')
print(f'Test     : {len(b_test_pos)} pos, {len(b_test_neg)} neg')

Training : 53 pos, 1341 neg
Test     : 14 pos, 336 neg


In [12]:
run_group_c(
    b_fluents, b_actions, b_static,
    b_train_pos, b_train_neg, b_test_pos, b_test_neg,
    label='Blocks domain (extended, 20 problems)'
)


=== Blocks domain (extended, 20 problems) ===
Training: 53 pos + 1341 neg (1394 total)
Test    : 14 pos + 336 neg
 Snap  Demo  evaluate() P_s    R_c  evaluate_repr() P_s    R_c
----------------------------------------------------------------------
    0    25  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.2857❌
    1    51  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    2    77  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.4286❌
    3   104  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.4286❌
    4   130  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    5   156  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    6   183  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    7   209  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    8   235  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.5714❌
    9   262  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.8571❌
   10   288  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.8571❌
   11   314  P_s=1.0000✓ R_c=1.0000✓  P_s=1.0000✓ R_c=0.8571❌
   12  

(53, 0)

## Final Boundary State — Blocks Domain

In [13]:
# Re-run a fresh learner on ALL blocks demos (no split) to inspect the final state.
full_learner = Learner(b_fluents, b_actions, b_static)

if not all_pos_blocks:
    all_train_b = list(all_neg_blocks)
elif not all_neg_blocks:
    all_train_b = list(all_pos_blocks)
else:
    slice_size  = len(all_neg_blocks) / len(all_pos_blocks)
    all_train_b = []
    for i, pos in enumerate(all_pos_blocks):
        all_train_b.extend(all_neg_blocks[int(slice_size * i):int(slice_size * (i + 1))])
        all_train_b.append(pos)

for d in all_train_b:
    full_learner.update(d)

print('=== Final boundaries — blocks domain (all demos) ===')
print(full_learner)

for a in b_actions:
    n      = a.name
    hL_pre = next(iter(full_learner.L_pre[n]))
    hL_eff = next(iter(full_learner.L_eff[n]))
    hU_eff = next(iter(full_learner.U_eff[n]))

    print(f'\nAction: {n}')
    print(f'  L_pre ({len(hL_pre)} lits)  : {sorted(str(l) for l in hL_pre)}')
    print(f'  U_pre ({len(full_learner.U_pre[n])} hyps): '
          + str([sorted(str(l) for l in h) for h in full_learner.U_pre[n]]))
    print(f'  L_eff ({len(hL_eff)} lits)  : {sorted(str(l) for l in hL_eff)}')
    print(f'  U_eff ({len(hU_eff)} lits)  : {sorted(str(l) for l in hU_eff)}')
    print(f'  Converged: {full_learner.version_space_size[n]["converged"]}')

=== Final boundaries — blocks domain (all demos) ===
Learner(actions=4, demos=1744, collapsed=0, converged=False)

Action: pickup
  L_pre (10 lits)  : ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)']
  U_pre (1 hyps): [['arm_empty()', 'clear(x)', 'on_table(x)']]
  L_eff (4 lits)  : ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬on_table(x)']
  U_eff (10 lits)  : ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)', '¬on_table(x)']
  Converged: False

Action: putdown
  L_pre (10 lits)  : ['holding(x)', '¬arm_empty()', '¬clear(x)', '¬holding(y)', '¬holding(z)', '¬on(x, y)', '¬on(x, z)', '¬on(y, x)', '¬on(z, x)', '¬on_table(x)']
  U_pre (1 hyps): [['holding(x)']]
  L_eff (4 lits)  : ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)']
  U_eff (10 lits)  : ['arm_empty()', 'clear(x)', 'on_table(x)', '¬holding(x)', '¬holding(y)